# Kumo Relational Quickstart with structured-data-models-client

**Kumo Relational** is a relational foundation model for
machine learning on enterprise data. With your relational data and a
Predictive Query Language (PQL) query, you can generate predictions without
training a task-specific model.

This quickstart uses four tables from Microsoft's **AdventureWorks** OLTP
sample database: customers, products, sales-order headers, and sales-order
details. Predictions are submitted to a deployed Kumo Relational NVIDIA NIM through
`structured-data-models-client`.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.

## Introduction

Kumo Relational is grounded in three key world views:

> **1. Enterprise data is a graph.**

Enterprise data is a graph where tables are connected by keys. In this notebook, `sales_order_headers` connects customers to orders, while `sales_order_details` connects each order to its products. Together, these relationships give Kumo Relational the business context surrounding every transaction.

Once we structure enterprise data this way, we can apply pre-trained Relational Graph Transformers to extract insights and patterns.

> **2. With timestamps, we place events on a timeline.**

By placing events on a timeline, we unlock the ability to model how things evolve. We can select any point in time and predict what is likely to happen next from the sequence and patterns in the historical data.

> **3. Machine learning tasks can be described via predictive queries.**

Major machine learning tasks, including regression, classification, forecasting, and recommendation, can be expressed using *Predictive Query Language (PQL)*. A PQL query identifies the outcome to predict, the entity to predict it for, and, for temporal tasks, the future prediction window.

If you know SQL, PQL should feel familiar. Learn more in the [query-writing guide](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/make-predictions/writing-predictive-queries).

**Let's get started!**

## Step 1. Install structured-data-models-client

Use a Snowflake notebook on **Container Runtime** with an active Snowpark
session. Configure an external access integration for PyPI package installation
and attach it to the notebook service. If the NIM is outside Snowflake, allow
its host through an external access integration too. See Snowflake's
[external access setup](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-external-access).
The notebook itself does not need a GPU; inference runs on the NIM service.

Install the relational and Snowflake extras. See
[Install and Connect](https://docs.nvidia.com/sdgm/client/kumo-relational/get-started/sdk-getting-started)
for supported Python versions and platforms. Restart the notebook kernel after
installation if these packages were already imported, then continue below.

In [ ]:
%pip install "structured-data-models-client[relational,snowflake]"

In [ ]:
import os

import pandas as pd
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    Relationship,
    Schema,
    SDMClient,
)
from sdm_connectors import SnowflakeSource

Build schemas and queries through `sdm_client`'s top-level names: `Schema`, `Relationship`, and `PredictiveQuery`. Submit predictions through `SDMClient`, naming the model with `KumoRelational()`.

## Step 2. Configure the Kumo Relational NIM endpoint

Deploy the NIM first using the
[Snowflake deployment guide](https://docs.nvidia.com/sdgm/client/kumo-relational/deployment/snowflake).
Set `KUMO_RELATIONAL_API_ENDPOINT` to its reachable address. The
`http://<dns_name>:8000` placeholder below is a private SPCS service address,
not a container running in the notebook.

For endpoints outside that private service network, use an `https://` URL with
TLS termination. Plain HTTP transmits table rows unencrypted. Set
`KUMO_RELATIONAL_API_KEY` only when an authenticating gateway protects the
endpoint; keep credentials in environment configuration.

In [ ]:
NIM_URL = os.environ.get(
    'KUMO_RELATIONAL_API_ENDPOINT',
    'http://<dns_name>:8000',
)
NIM_API_KEY = os.environ.get('KUMO_RELATIONAL_API_KEY')

## Step 3. Connect to the NIM

An `SDMClient` owns the HTTP connection to one NIM endpoint. Keep credentials
in environment configuration rather than notebook source.

In [ ]:
client = SDMClient(
    url=NIM_URL,
    api_key=NIM_API_KEY,
)

print('Ready:', client.health_ready())
print('Capabilities:', client.capabilities(KumoRelational()))

<div class="alert alert-warning">
<strong>Security:</strong> Never hard-code NIM or gateway credentials in a
notebook. Use environment variables or your platform's secret manager.
</div>

## Step 4. Prepare the Snowflake tables

Download `Customer.csv`, `Product.csv`, `SalesOrderHeader.csv`, and
`SalesOrderDetail.csv` from the pinned
[Microsoft AdventureWorks OLTP sample files](https://github.com/microsoft/sql-server-samples/tree/b47eadc85277f72843647223cde53aa1b85ffcb8/samples/databases/adventure-works/oltp-install-script).
These files are **tab-separated and have no header row**, despite their `.csv`
extension. The table definitions below provide the column names in file order.

Upload the four files yourself to an internal Snowflake stage. In a SQL
worksheet, replace `<database>`, `<schema>`, and `<stage>` with your destination
names, then create and load these tables in a dedicated schema. The quoted
identifiers preserve the capitalization used by the notebook. Your role needs
access to the database, schema, warehouse, and stage, permission to create and
load these tables, and permission to create the context view used below.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
Microsoft's samples are provided under the
[MIT License](https://github.com/microsoft/sql-server-samples/blob/b47eadc85277f72843647223cde53aa1b85ffcb8/license.txt).

```sql
CREATE TABLE IF NOT EXISTS <database>.<schema>."Customer" (
  "CustomerID" INTEGER,
  "PersonID" INTEGER,
  "StoreID" INTEGER,
  "TerritoryID" INTEGER,
  "AccountNumber" VARCHAR,
  "rowguid" VARCHAR,
  "ModifiedDate" TIMESTAMP_NTZ
);

CREATE TABLE IF NOT EXISTS <database>.<schema>."Product" (
  "ProductID" INTEGER,
  "Name" VARCHAR,
  "ProductNumber" VARCHAR,
  "MakeFlag" INTEGER,
  "FinishedGoodsFlag" INTEGER,
  "Color" VARCHAR,
  "SafetyStockLevel" INTEGER,
  "ReorderPoint" INTEGER,
  "StandardCost" FLOAT,
  "ListPrice" FLOAT,
  "Size" VARCHAR,
  "SizeUnitMeasureCode" VARCHAR,
  "WeightUnitMeasureCode" VARCHAR,
  "Weight" FLOAT,
  "DaysToManufacture" INTEGER,
  "ProductLine" VARCHAR,
  "Class" VARCHAR,
  "Style" VARCHAR,
  "ProductSubcategoryID" INTEGER,
  "ProductModelID" INTEGER,
  "SellStartDate" TIMESTAMP_NTZ,
  "SellEndDate" TIMESTAMP_NTZ,
  "DiscontinuedDate" TIMESTAMP_NTZ,
  "rowguid" VARCHAR,
  "ModifiedDate" TIMESTAMP_NTZ
);

CREATE TABLE IF NOT EXISTS <database>.<schema>."SalesOrderHeader" (
  "SalesOrderID" INTEGER,
  "RevisionNumber" INTEGER,
  "OrderDate" TIMESTAMP_NTZ,
  "DueDate" TIMESTAMP_NTZ,
  "ShipDate" TIMESTAMP_NTZ,
  "Status" INTEGER,
  "OnlineOrderFlag" INTEGER,
  "SalesOrderNumber" VARCHAR,
  "PurchaseOrderNumber" VARCHAR,
  "AccountNumber" VARCHAR,
  "CustomerID" INTEGER,
  "SalesPersonID" INTEGER,
  "TerritoryID" INTEGER,
  "BillToAddressID" INTEGER,
  "ShipToAddressID" INTEGER,
  "ShipMethodID" INTEGER,
  "CreditCardID" INTEGER,
  "CreditCardApprovalCode" VARCHAR,
  "CurrencyRateID" INTEGER,
  "SubTotal" FLOAT,
  "TaxAmt" FLOAT,
  "Freight" FLOAT,
  "TotalDue" FLOAT,
  "Comment" VARCHAR,
  "rowguid" VARCHAR,
  "ModifiedDate" TIMESTAMP_NTZ
);

CREATE TABLE IF NOT EXISTS <database>.<schema>."SalesOrderDetail" (
  "SalesOrderID" INTEGER,
  "SalesOrderDetailID" INTEGER,
  "CarrierTrackingNumber" VARCHAR,
  "OrderQty" INTEGER,
  "ProductID" INTEGER,
  "SpecialOfferID" INTEGER,
  "UnitPrice" FLOAT,
  "UnitPriceDiscount" FLOAT,
  "LineTotal" FLOAT,
  "rowguid" VARCHAR,
  "ModifiedDate" TIMESTAMP_NTZ
);

COPY INTO <database>.<schema>."Customer"
  FROM @<database>.<schema>.<stage>/Customer.csv
  FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = '\t' SKIP_HEADER = 0
                 FIELD_OPTIONALLY_ENCLOSED_BY = NONE NULL_IF = (''));

COPY INTO <database>.<schema>."Product"
  FROM @<database>.<schema>.<stage>/Product.csv
  FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = '\t' SKIP_HEADER = 0
                 FIELD_OPTIONALLY_ENCLOSED_BY = NONE NULL_IF = (''));

COPY INTO <database>.<schema>."SalesOrderHeader"
  FROM @<database>.<schema>.<stage>/SalesOrderHeader.csv
  FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = '\t' SKIP_HEADER = 0
                 FIELD_OPTIONALLY_ENCLOSED_BY = NONE NULL_IF = (''));

COPY INTO <database>.<schema>."SalesOrderDetail"
  FROM @<database>.<schema>.<stage>/SalesOrderDetail.csv
  FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = '\t' SKIP_HEADER = 0
                 FIELD_OPTIONALLY_ENCLOSED_BY = NONE NULL_IF = (''));
```

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session

session = get_active_session()
DATABASE = '<database>'
SCHEMA = '<schema>'
DATA_LOCATION = f'"{DATABASE}"."{SCHEMA}"'

Create a temporary view that joins each order line to its parent order's
`CustomerID` and `OrderDate`. The uploaded source tables stay unchanged.
`SnowflakeSource` uses this same session, so the view remains visible when the
client samples context. Rerun this cell after restarting the session.

In [ ]:
session.sql(f"""CREATE OR REPLACE TEMPORARY VIEW
    {DATA_LOCATION}."SDM_SalesOrderDetail" AS
SELECT d.*, h."CustomerID", h."OrderDate"
FROM {DATA_LOCATION}."SalesOrderDetail" AS d
JOIN {DATA_LOCATION}."SalesOrderHeader" AS h
  ON d."SalesOrderID" = h."SalesOrderID"
""").collect()

## Step 5. Build a schema from Snowflake

`Schema.from_source()` describes the tables where they are stored. It reads
metadata and a small preview to infer types, and checks the configured keys in
Snowflake. Prediction requests sample related rows with SQL rather than copying
all four source tables into pandas. Review the inferred semantic types before
prediction.

In [ ]:
source = SnowflakeSource(session, database=DATABASE, schema=SCHEMA)
schema = Schema.from_source(
    source,
    {
        'customers': 'Customer',
        'products': 'Product',
        'sales_order_headers': 'SalesOrderHeader',
        'sales_order_details': 'SDM_SalesOrderDetail',
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_headers': 'SalesOrderID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        # Customer and product attributes have no event timeline here.
        'customers': None,
        'products': None,
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
    },
    relationships=[
        Relationship(
            source_table='sales_order_headers',
            source_column='CustomerID',
            target_table='customers',
            target_column='CustomerID',
        ),
        Relationship(
            source_table='sales_order_details',
            source_column='SalesOrderID',
            target_table='sales_order_headers',
            target_column='SalesOrderID',
        ),
        Relationship(
            source_table='sales_order_details',
            source_column='ProductID',
            target_table='products',
            target_column='ProductID',
        ),
        Relationship(
            source_table='sales_order_details',
            source_column='CustomerID',
            target_table='customers',
            target_column='CustomerID',
        ),
    ],
)

print(schema)
for name in schema.tables:
    print(schema[name])

Keys and relationship columns use the `ID` semantic type. The next cell
omits identifier-like strings, free-text comments, and fields that can reveal
information recorded after an anchor, including modification times, ship dates,
and product sell dates. Product `Name` remains categorical and `OrderQty` is
numerical. See
[Data Types](https://docs.nvidia.com/sdgm/client/kumo-relational/user-guide/prepare-data/data-types).

This removes known leakage-prone fields; it does not reconstruct earlier values
of the remaining untimed customer and product attributes.

In [ ]:
leave_out = {
    'customers': ['rowguid', 'ModifiedDate', 'AccountNumber'],
    'products': [
        'rowguid',
        'ModifiedDate',
        'ProductNumber',
        'SellStartDate',
        'SellEndDate',
        'DiscontinuedDate',
    ],
    'sales_order_headers': [
        'rowguid',
        'ModifiedDate',
        'SalesOrderNumber',
        'PurchaseOrderNumber',
        'AccountNumber',
        'CreditCardApprovalCode',
        'Comment',
        'RevisionNumber',
        'Status',
        'ShipDate',
    ],
    'sales_order_details': ['rowguid', 'ModifiedDate', 'CarrierTrackingNumber'],
}
for table_name, column_names in leave_out.items():
    for column_name in column_names:
        del schema[table_name][column_name]

schema['products']['Name'].stype = 'categorical'
schema['sales_order_details']['OrderQty'].stype = 'numerical'

Call `schema.check_consistent()` after all metadata changes. An incompatible
semantic type, a missing column, or a primary key that is also configured as
a time column raises an error.

In [ ]:
schema.check_consistent()
print('Schema OK.')
for name in schema.tables:
    print(schema[name])

## Step 6. Submit prediction requests

The schema is a reusable description of the data. For each prediction:

1. Write the task in PQL.
2. Build a `PredictiveQuery` from the query text and the schema.
3. Call `.materialize(indices=...)` to gather a `Request`.
4. Call `client.predict(request, model=KumoRelational())`.

The result's `to_pandas()` method returns a pandas DataFrame.

**Note:** AdventureWorks is sample data, and the queries and results below are intended for demonstration. Benchmark Kumo Relational on data representative of your own use case before drawing performance conclusions.

#### PQL: How you describe a prediction task

Learn more in the [query-writing guide](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/make-predictions/writing-predictive-queries).

The short reference below covers the entity, target, aggregation,
prediction-window, and condition syntax used by the examples in this
quickstart.

<details>
<summary><b>💡 Click here for a short introduction to Predictive Query!</b></summary>

**Entities** can be specified in three ways:

* `FOR <entity_table>.<entity_primary_key>=1` embeds one entity ID in PQL.
* `FOR <entity_table>.<entity_primary_key> IN (1, 2, 3)` embeds several
  entity IDs in PQL.
* `FOR EACH <entity_table>.<entity_primary_key>` declares the entity column,
  while `materialize(indices=[1, 2, 3])` supplies the entity IDs separately.

**Preferred:** For application code, use `FOR EACH` and pass primary-key values
through `materialize()`'s `indices` argument. This keeps the prediction task separate
from the entity batch and makes the same PQL reusable. Despite its name,
`indices` contains source primary-key values, not DataFrame row numbers.
If `indices` is provided, it overrides entity IDs embedded in the PQL.

**Targets** can be specified in two common ways:

* `PREDICT <target_table>.<target_column>` imputes a missing value.
* `PREDICT <aggregation>(<target>, <start_offset>, <end_offset>, <time_unit>)` predicts an aggregate in a future window.

Supported aggregations include `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`, and `LIST_DISTINCT`. For example, `PREDICT COUNT(sales_order_headers.*, 0, 7, days)` predicts the number of orders in the next seven days.

Conditions can turn an aggregate into an event. For example, appending `=0` predicts the probability that no orders occur in the specified window.
</details>

### Example 1A: Forecast 30-day product demand

Predict how many units of product `ProductID=870` will be ordered in the next 30 days.

In [ ]:
query = PredictiveQuery(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) FOR EACH products.ProductID',
    schema,
)

request = query.materialize(indices=[870])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the product with `ProductID=870`
2. `prediction`: the predicted number of units ordered during the next 30 days

This result can support inventory and demand planning.

### Example 1B: Forecast from a historical anchor time

By default, predictions use the maximum timestamp in the temporal schema.
Set a historical `anchor_time` to make a prediction relative to an earlier date.
Temporal sampling uses the configured `OrderDate` columns for sales-order
headers and details to restrict their records to the anchor time.

The `customers` and `products` tables have no time column in this schema, so
their current attributes remain available at every anchor. Setting an anchor
does not recover earlier attribute values. Use historical snapshots of those
tables for a point-in-time evaluation.

In [ ]:
# Query only the latest timestamp rather than loading the order table.
latest_order = (
    session.table(f'{DATA_LOCATION}."SalesOrderHeader"')
    .select(F.max(F.col('"OrderDate"')))
    .collect()[0][0]
)
anchor_time = pd.Timestamp(latest_order) - pd.Timedelta(days=60)

request = query.materialize(indices=[870], anchor_time=anchor_time)
result = client.predict(request, model=KumoRelational())
result.to_pandas()

### Example 1C: Forecast for multiple steps

In [ ]:
query_multi_step = PredictiveQuery(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
    schema,
)

request = query_multi_step.materialize(indices=[870])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

A multi-step forecast returns one row per timeframe.
`forecast_step` is the one-based sequence number and `prediction` is the
forecast value for that period.

### Example 2: Predict customer inactivity

Predict the likelihood that customers `CustomerID=11176` and `CustomerID=11091` will place zero orders in the next 90 days.

In [ ]:
query_inactivity = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days)=0 FOR EACH customers.CustomerID',
    schema,
)

request = query_inactivity.materialize(indices=[11176, 11091])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the customer represented by each `CustomerID`
2. `prediction`: whether the zero-order event is predicted to occur
3. `probability_*` columns: the estimated likelihood of each outcome

This result can support retention or re-engagement workflows.

### Example 3: Recommend products

Rank which of a small candidate set of products customer `CustomerID=11176` is most likely to order in the next 30 days.

Ranking runs as one classification call per candidate, so the candidate set is
named explicitly with `candidates=` rather than left for the client to infer.
This example candidates the 15 most frequently ordered products so each has
enough purchase history for the model to learn from; an arbitrary or rarely
ordered candidate set can fail with "nothing to rank" because its context
holds only one outcome. See
[Prediction Types](https://docs.nvidia.com/sdgm/kumo-relational/user-guide/make-predictions/prediction-types#recommendations-ranking).

In [ ]:
# Collect only the 15 candidate IDs; aggregate purchase counts in Snowflake.
candidate_product_ids = (
    session.table(f'{DATA_LOCATION}."SalesOrderDetail"')
    .group_by(F.col('"ProductID"'))
    .agg(F.count('*').alias('purchase_count'))
    .sort(F.col('purchase_count').desc(), F.col('"ProductID"'))
    .limit(15)
    .to_pandas()['ProductID']
)

query_rank = PredictiveQuery(
    'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
    'RANK TOP 10 FOR EACH customers.CustomerID',
    schema,
)

request = query_rank.materialize(
    indices=[11176],
    candidates=candidate_product_ids,
    num_context_samples=5000,
)
result = client.predict(request, model=KumoRelational())
print(result.to_pandas())
print('Unscored candidates:', result.unscored_candidates)

How to interpret the result:

1. `id`: the customer with `CustomerID=11176`
2. `rankings`: the ranked candidates, each with a `candidate` product ID and a `score`

A candidate whose sampled context holds only one outcome can't be scored and
is left out rather than failing the whole ranking; check
`result.unscored_candidates` for which ones were skipped.

### Example 4: Infer a missing product attribute

Predict the missing `Color` value for product `ProductID=870`.

In [ ]:
query_attribute = PredictiveQuery(
    'PREDICT products.Color FOR EACH products.ProductID',
    schema,
)

request = query_attribute.materialize(indices=[870])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the product with `ProductID=870`
2. `prediction`: the predicted color category
3. `probability_*` columns: the estimated likelihood of each category

Attribute inference can help enrich incomplete catalog records, but predicted values should be reviewed before being written back to a source system.

### Example 5: Predict customer churn

Predict the likelihood that customers `CustomerID=11176` and `CustomerID=11091` will place zero orders in the next 90 days, reusing Example 2's query with one addition. The `WHERE` condition does not filter which entities are scored, both `indices` are still predicted; it filters which rows qualify as labeled context examples, restricting the model's learned comparison to customers who themselves placed at least one order in the previous 90 days.

In [ ]:
query_churn = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID '
    'WHERE COUNT(sales_order_headers.*, -90, 0, days) > 0',
    schema,
)

request = query_churn.materialize(indices=[11176, 11091])
result = client.predict(request, model=KumoRelational())
result.to_pandas()

How to interpret the result:

1. `id`: the customer represented by each `CustomerID`
2. `prediction`: whether the customer is predicted to place zero orders during the next 90 days
3. `probability_*` columns: the estimated likelihood of each outcome

This churn prediction can support retention or re-engagement workflows.

## Step 7. Close the client

Close the client when the notebook is finished to release its HTTP connection.

In [ ]:
client.close()